In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
os.chdir("../")

In [3]:
from ease_recommender import *
from npmi_recommender import *

import pickle as p

In [4]:
def create_mat(row, col, bool_to_int=True):
    # bool_to_int won't count duplicates in the same row, creates a different weighting basically
    if bool_to_int:
        data = np.ones_like(row, dtype=bool)
        return csr_matrix((data, (row, col))).astype(np.int64)
    else:
        data = np.ones_like(row, dtype=np.int64)
        return csr_matrix((data, (row, col)))

def check_if_all_terms_in_str(q, terms):
    for term in terms:
        if term not in q:
            return False

    return True

def get_cat2idx(category_type, D):
    if category_type == "track":
        return D["track2idx"]
    elif category_type == "album":
        return D["album2idx"]
    elif category_type == "artist":
        return D["artist2idx"]
    else:
        raise NotImplementedError

def find_match_using_terms(terms, cat2idx):
    matches = []
    for name in cat2idx.keys():
        if check_if_all_terms_in_str(name, terms):
            matches.append(name)

    if len(matches) > 1:
        raise Exception("Multiple matches found, filter down to a single match", matches)

    return matches[0]

In [5]:
print("loading cache data...")
D = p.load(open("cached_data/spotify_preprocessed.p", "rb"))

print("building csr matrices...")

# TODO: finish implementing track and album level recommendations

# track_mat = create_mat(D["playlist_indices"], D["track_indices"])
# album_mat = create_mat(D["playlist_indices"], D["album_indices"])
artist_mat = create_mat(D["playlist_indices"], D["artist_indices"])

print("done")

loading cache data...
building csr matrices...
done


In [6]:
cat2idx = get_cat2idx("artist", D)
idx2cat = {v:k for k, v in cat2idx.items()}

In [7]:
def get_item_idx(terms):
    assert type(terms) == list
    
    a_name = find_match_using_terms(terms, cat2idx)
    a = cat2idx[a_name]
    
    print("Artist:", a_name)
    print("*" * 20)
    
    assert artist_mat[:, a].sum() > 0
    
    return a

In [8]:
mat = artist_mat
mat = csr_array(mat)

In [9]:
mat.shape

(1000000, 295860)

In [10]:
X = mat.T @ mat

In [11]:
X.shape

(295860, 295860)

In [60]:
import numpy as np
from scipy import sparse
from scipy.optimize import minimize

class SppmiOptimizerCache:
    """Caches global raw matrix invariants in memory for rapid lookup loops."""
    def __init__(self, X):
        self.X_csc = X.tocsc() if not sparse.isspmatrix_csc(X) else X
        self.rows, self.cols = self.X_csc.shape
        self.N_raw = self.X_csc.sum()
        self.row_sums_raw = np.array(self.X_csc.sum(axis=1)).flatten()
        self.col_sums_raw = np.array(self.X_csc.sum(axis=0)).flatten()


def compute_item_scores(cache, a, alpha, gamma, tau, zero_diag=True):
    """Generates the full 1D similarity profile vector for an item 'a'."""
    rows, cols = cache.rows, cache.cols
    N_smoothed = cache.N_raw + (alpha * rows * cols)
    
    col_a = cache.X_csc[:, a:a+1]
    curr_rows, curr_data = col_a.indices, col_a.data.astype(np.float32)
    
    P_x = (cache.row_sums_raw + (alpha * cols)) / N_smoothed
    P_y_a = (cache.col_sums_raw[a] + (alpha * rows)) / N_smoothed
    
    scores = np.zeros(rows, dtype=np.float32)
    
    # SHORT-CIRCUIT OPTIMIZATION: If tau is disabled but alpha is 0, 
    # all structural zeros are 0.0 anyway. We can skip full dense grid calculations.
    if tau > 0 or alpha == 0:
        P_xy_nz = (curr_data + alpha) / N_smoothed
        scores_nz = P_xy_nz / ((P_x[curr_rows] ** gamma) * (P_y_a ** gamma))
        if tau > 0:
            scores_nz *= (curr_data / (curr_data + tau))
        scores[curr_rows] = scores_nz
    else:
        # Full dense field processing (only active if alpha > 0 AND tau == 0)
        P_xy = np.full(rows, alpha / N_smoothed, dtype=np.float32)
        P_xy[curr_rows] = (curr_data + alpha) / N_smoothed
        scores = P_xy / ((P_x ** gamma) * (P_y_a ** gamma))
        
    if zero_diag:
        scores[a] = 0.0
        
    return scores


def optimize_sppmi_parameters(cache, item_groups, init_params=None, optimize_vars=None):
    """
    Optimizes selective SPPMI parameters concurrently using Powell's method.
    Bypasses and neutralizes parameters explicitly set to None.
    """
    default_params = {'alpha': 0.1, 'gamma': 1.0, 'tau': 2.0}
    neutral_defaults = {'alpha': 0.0, 'gamma': 1.0, 'tau': 0.0}

    # Clean input dictionaries
    if init_params is None:
        init_params = default_params.copy()
    else:
        for k, v in default_params.items():
            if k not in init_params:
                init_params[k] = v

    if optimize_vars is None:
        optimize_vars = ['alpha', 'gamma', 'tau']

    # CRITICAL: Validate and substitute None values with neutral defaults
    for var in ['alpha', 'gamma', 'tau']:
        if init_params[var] is None:
            assert var not in optimize_vars, f"AssertionError: Cannot optimize '{var}' because it is set to None."
            init_params[var] = neutral_defaults[var]

    for i, group in enumerate(item_groups):
        assert len(group) >= 2, f"Group {i} has fewer than 2 items. Cannot compute mutual ranks."

    unique_items = list(set(idx for group in item_groups for idx in group))
    x0 = [init_params[var] for var in optimize_vars]

    def objective(x_vec):
        current_params = init_params.copy()
        for var_name, val in zip(optimize_vars, x_vec):
            current_params[var_name] = val
            
        alpha, gamma, tau = current_params['alpha'], current_params['gamma'], current_params['tau']
        
        # Hard boundary protection penalties to block line searches into illegal spaces
        # (Allows alpha to be exactly 0.0 now if it was intentionally passed as None)
        if alpha < 0.0 or gamma <= 0.0 or tau < 0.0:
            return 1e9
            
        current_scores = {idx: compute_item_scores(cache, idx, alpha, gamma, tau) for idx in unique_items}
        group_scores = []
        
        for group in item_groups:
            total_pairwise_rank = 0.0
            k = len(group)
            
            for u in group:
                sort_indices = np.argsort(-current_scores[u])
                ranks = np.empty_like(sort_indices)
                ranks[sort_indices] = np.arange(len(sort_indices))
                
                for v in group:
                    if u == v:
                        continue
                    total_pairwise_rank += ranks[v]
                    
            group_scores.append(total_pairwise_rank / (k * (k - 1)))
            
        return np.mean(group_scores)

    # Execute multidimensional optimization if there are active variables to track
    if len(optimize_vars) > 0:
        res = minimize(objective, x0, method='Powell')
        final_values = res.x if len(optimize_vars) > 1 else [res.x]
        
        final_params = init_params.copy()
        for var_name, val in zip(optimize_vars, final_values):
            final_params[var_name] = float(val)
        return res.fun, final_params
    else:
        # Short circuit optimization if all variables are locked/None
        return objective(init_params), init_params

In [13]:
cache = SppmiOptimizerCache(X)

In [14]:
# terms = ["Megadeth"]
# terms = ["Havok (spotify:artist:2jw4wgixxa20jls9N3Bdpq)"]
# terms = ["Evile (spotify:artist:1dwrMJAKBiLlj0O4R791Xo)"]
# terms = ["Sonata Arctica"]
# terms = ["Muse (spotify:artist:12Chz98pHFMPJEknJQMWvI)"]
# terms = ["Dream Theater (spotify:artist:2aaLAng2L2aWD2FClzwiep)"]
# terms = ["Saints Go Machine"]
# terms = ["AURORA (spotify:artist:1WgXqy2Dd70QQOU7Ay074N)"]
# terms = ["Gabrielle (spotify:artist:4OovmAu23KrDlDQI2UbneL)"]
# terms = ["Bon Iver (spotify:artist:4LEiUm1SRbFMgfqnQTwUbQ)"]
# terms = ["Fleet Foxes"]
# terms = ["Vektor (spotify:artist:09mNj9XgCqgg6usfeXOoBg)"]
# terms = ["Dr. Living Dead (spotify:artist:0gLz6azFpZgHyJkJd5yuiM)"]
# terms = ["Lich King (spotify:artist:4rlxS0LeVnHz6z1zp2iJbz)"]
# terms = ["Skeletonwitch (spotify:artist:213mmq3zkNWx7CtfzftTC5)"]
# terms = ["Wintersun (spotify:artist:6ui6SwChan7c1KYBQCqGKV)"]
# terms = ["Chris Poland"]
# terms = ["Marty Friedman (spotify:artist:5czW6bitDSKbNBNDizRT9p)"]
# terms = ["Jason Becker (spotify:artist:0A4Z1qNp3lGWa9VI66M67D)"]
# terms = ["Cacophony (spotify:artist:3WNx4M2YbMmDiJqeOBi0Ae)"]

# a = get_item_idx(["Cacophony (spotify:artist:3WNx4M2YbMmDiJqeOBi0Ae)"])
# b = get_item_idx(["Jason Becker (spotify:artist:0A4Z1qNp3lGWa9VI66M67D)"])

In [49]:
item_group_a = [
    get_item_idx(["Highasakite"]),
    get_item_idx(["Ásgeir (spotify:artist:7xUZ4069zcyBM4Bn10NQ1c)"]),
]

Artist: Highasakite (spotify:artist:5awQWdBpLqN2KFVRN8w56T)
********************
Artist: Ásgeir (spotify:artist:7xUZ4069zcyBM4Bn10NQ1c)
********************


In [50]:
item_group_b = [
    get_item_idx(["Cacophony (spotify:artist:3WNx4M2YbMmDiJqeOBi0Ae)"]),
    get_item_idx(["Jason Becker (spotify:artist:0A4Z1qNp3lGWa9VI66M67D)"]),
#     get_item_idx(["Marty Friedman (spotify:artist:5czW6bitDSKbNBNDizRT9p)"]),
]

Artist: Cacophony (spotify:artist:3WNx4M2YbMmDiJqeOBi0Ae)
********************
Artist: Jason Becker (spotify:artist:0A4Z1qNp3lGWa9VI66M67D)
********************


In [54]:
# nested_items = [
#     item_group_a,
#     item_group_b,
# ]

In [55]:
nested_items = [
    item_group_a,
]

In [61]:
# Optimize everything at once
best_score, best_params = optimize_sppmi_parameters(
    cache, 
    item_groups=nested_items,
    init_params={'alpha': 0.06397197616493325, 'gamma': 1.2987089696156853, 'tau': 13.338553077819997},
#     optimize_vars=['alpha', 'gamma', 'tau'] # <- Tells the engine to tune all three
    optimize_vars=[] # <- Tells the engine to tune all three
)

print("Optimized Parameters:", best_params)
print("Best Achieved Mean Rank:", best_score)

Optimized Parameters: {'alpha': 0.06397197616493325, 'gamma': 1.2987089696156853, 'tau': 13.338553077819997}
Best Achieved Mean Rank: 5.0


In [47]:
# Optimize everything at once
res, best_params = optimize_sppmi_parameters(
    cache, 
    item_groups=nested_items,
    init_params={'alpha': 0.117320, 'gamma': 1.4, 'tau':4.4},
    optimize_vars=['alpha', 'gamma', 'tau'] # <- Tells the engine to tune all three
)

print("Optimized Parameters:", best_params)
print("Best Achieved Mean Rank:", res.fun)

Optimized Parameters: {'alpha': 0.11732000001246078, 'gamma': 1.3879611044586408, 'tau': 4.775388190607372}
Best Achieved Mean Rank: 3.75


In [48]:
# Optimize everything at once
res, best_params = optimize_sppmi_parameters(
    cache, 
    item_groups=nested_items,
    init_params={'alpha': 0, 'gamma': 1, 'tau':0},
#     optimize_vars=['alpha', 'gamma', 'tau'] # 4.0
#     optimize_vars=['gamma', 'alpha', 'tau'] # 3.75
    optimize_vars=['tau', 'gamma', 'alpha'] # 3.25
#     optimize_vars=['tau', 'alpha', 'gamma'] # 4.75
#     optimize_vars=['gamma', 'tau', 'alpha'] # 5.75 
#     optimize_vars=['alpha', 'tau', 'gamma'] # 4.0
    
#     optimize_vars=['tau', 'gamma'] # 34.0
#     optimize_vars=['gamma', 'tau'] # 23.0
#     optimize_vars=['alpha', 'tau'] # 5.5
#     optimize_vars=['tau', 'alpha'] # 5.25
#     optimize_vars=['alpha', 'gamma'] # 5.5
#     optimize_vars=['gamma', 'alpha'] # 5.5
    
#     optimize_vars=['alpha'] # 5.5
#     optimize_vars=['gamma'] # 59.0
#     optimize_vars=['tau'] # 66.5
)

print("Optimized Parameters:", best_params)
print("Best Achieved Mean Rank:", res.fun)

Optimized Parameters: {'alpha': 0.06397197616493325, 'gamma': 1.2987089696156853, 'tau': 13.338553077819997}
Best Achieved Mean Rank: 3.25


In [ ]:
# Pass parameters using dictionary unpacking (**best_params)
new_scores = compute_item_scores(cache, a=999, **best_params)

top_recs = np.argsort(-new_scores)[:5]
print("Top recommendations for item 999:", top_recs)

In [15]:
# 1. Initialize the global shared data structure
cache = SppmiOptimizerCache(X)

In [16]:
# items = [
#     get_item_idx(["Cacophony (spotify:artist:3WNx4M2YbMmDiJqeOBi0Ae)"]),
#     get_item_idx(["Jason Becker (spotify:artist:0A4Z1qNp3lGWa9VI66M67D)"]),
# #     get_item_idx(["Marty Friedman (spotify:artist:5czW6bitDSKbNBNDizRT9p)"]),
# ]

In [66]:
ranking_type = "soft_power_lift"
gamma = 1.4
tau = 4.4

# 2. STEP 1: Fit parameters using your validation pairs (a=42, b=107)
fit_result = optimize_alpha_nested_groups(
    cache,
    nested_items,
    ranking_type=ranking_type, 
    gamma=gamma, 
    tau=tau,
    return_scores=False
)

# Extract your optimal fitted alpha configuration
best_alpha = fit_result.x
print(f"--- Fit Complete ---")
print(f"Target Optimized Alpha: {best_alpha:.6f}\n")
print(fit_result.fun)

--- Fit Complete ---
Target Optimized Alpha: 0.117320

3.75


In [ ]:
a, b = item_group_a

scores_a = compute_item_scores(
    cache, 
    a=a, 
    alpha=best_alpha, 
    ranking_type=ranking_type,
    gamma=gamma, 
    tau=tau,
)

scores_b = compute_item_scores(
    cache, 
    a=b, 
    alpha=best_alpha, 
    ranking_type=ranking_type,
    gamma=gamma, 
    tau=tau,
)

(np.argsort(-scores_a).tolist().index(b) + np.argsort(-scores_b).tolist().index(a))/2

In [124]:
k = 15

# Asgeir
# terms = ["Ásgeir (spotify:artist:7xUZ4069zcyBM4Bn10NQ1c)"]
terms = ["Highasakite"]
# terms = ["Elsa & Emilie"]
# terms = ["Susanne Sundfør"]
# terms = ["Billy Joel (spotify:artist:6zFYqv1mOsgBRQbae3JJ9e)"]
# terms = ["Megadeth"]
# terms = ["Havok (spotify:artist:2jw4wgixxa20jls9N3Bdpq)"]
# terms = ["Evile (spotify:artist:1dwrMJAKBiLlj0O4R791Xo)"]
# terms = ["Sonata Arctica"]
# terms = ["Muse (spotify:artist:12Chz98pHFMPJEknJQMWvI)"]
# terms = ["Dream Theater (spotify:artist:2aaLAng2L2aWD2FClzwiep)"]
# terms = ["Saints Go Machine"]
# terms = ["AURORA (spotify:artist:1WgXqy2Dd70QQOU7Ay074N)"]
# terms = ["Gabrielle (spotify:artist:4OovmAu23KrDlDQI2UbneL)"]
# terms = ["Bon Iver (spotify:artist:4LEiUm1SRbFMgfqnQTwUbQ)"]
# terms = ["Fleet Foxes"]
# terms = ["Vektor (spotify:artist:09mNj9XgCqgg6usfeXOoBg)"]
# terms = ["Dr. Living Dead (spotify:artist:0gLz6azFpZgHyJkJd5yuiM)"]
# terms = ["Lich King (spotify:artist:4rlxS0LeVnHz6z1zp2iJbz)"]
# terms = ["Skeletonwitch (spotify:artist:213mmq3zkNWx7CtfzftTC5)"]
# terms = ["Wintersun (spotify:artist:6ui6SwChan7c1KYBQCqGKV)"]
# terms = ["Chris Poland"]
# terms = ["Marty Friedman (spotify:artist:5czW6bitDSKbNBNDizRT9p)"]
# terms = ["Jason Becker (spotify:artist:0A4Z1qNp3lGWa9VI66M67D)"]
# terms = ["Cacophony (spotify:artist:3WNx4M2YbMmDiJqeOBi0Ae)"]

item_idx = get_item_idx(terms)

scores = compute_item_scores(
    cache, 
    a=item_idx, 
    alpha=best_alpha, 
    ranking_type=ranking_type,
    gamma=gamma, 
    tau=tau,
)

for i in np.argsort(-scores)[:k]:
    print(idx2cat[i])

Artist: Highasakite (spotify:artist:5awQWdBpLqN2KFVRN8w56T)
********************
Gabrielle (spotify:artist:4OovmAu23KrDlDQI2UbneL)
Elsa & Emilie (spotify:artist:4HDNQLqhooVfWXtIRMyqMY)
Susanne Sundfør (spotify:artist:54KCNI7URCrG6yjQK3Ukow)
Nils Bech (spotify:artist:57QhXfAsLsIRtgC1VfHu1F)
Rockettothesky (spotify:artist:0nu7qEOc8X8UFK10d8lsLw)
Ings (spotify:artist:3wJ2NeHF58Im2tojNX8ESR)
Alice Boman (spotify:artist:3WiytRnvoL0kT3oAGl9TCt)
Kjartan Lauritzen (spotify:artist:0TW5M8RYADmgeCP1q523hf)
Ásgeir (spotify:artist:7xUZ4069zcyBM4Bn10NQ1c)
Amason (spotify:artist:4cJKxS7uOPhwb5UQ70sYpN)
Karpe Diem (spotify:artist:3X23gpg1vPacr0hBARyxtN)
Majical Cloudz (spotify:artist:4BEYBN6NCPrFk3sOLMTby3)
SOAK (spotify:artist:4PLsMEk2DCRVlVL2a9aZAv)
Phoria (spotify:artist:0HDxlFsXwyrpufs4YgTNMm)
Oh Pep! (spotify:artist:3L9rqEIsNSaOcx2QIstn7v)
